## **DO NOT rename or change the signature of these functions. Your code must be in the 3rd cell of the notebook, otherwise the tests will fall.**

# Homework: AI Agents

## Instructions
1. **"Template" cell** — run it first, do not modify.
2. **"Tasks" cell** — write your code where you see `# YOUR CODE HERE`.
3. Run the open examples and make sure all say `OK`.
4. Submit the notebook with saved outputs.

In [1]:
import os
MODEL_NAME = "gpt-oss-20b"
YANDEX_CLOUD_FOLDER = "..."

os.environ['OPENAI_API_KEY'] = '...'
os.environ['OPENAI_BASE_URL'] = "https://ai.api.cloud.yandex.net/v1"
os.environ["OPENAI_MODEL"] = f"gpt://{YANDEX_CLOUD_FOLDER}/{MODEL_NAME}"

In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          TEMPLATE — DO NOT MODIFY THIS CELL                 ║
# ╚══════════════════════════════════════════════════════════════╝
# %pip install -q langchain-openai langchain-core

import os, json, copy
from typing import Any
from pathlib import Path
from dataclasses import dataclass, field

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool

MODEL_NAME = "gpt-oss-20b"
MODEL_NAME = f"gpt://{YANDEX_CLOUD_FOLDER}/{MODEL_NAME}"
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "YOUR_KEY_HERE")
llm = ChatOpenAI(model=MODEL_NAME, temperature=0)


def llm_chat(messages: list, tools: list | None = None) -> AIMessage:
    """
    Sends the message history to the LLM and returns the model response.

    Parameters:
      messages — list of dialog messages. Each message is a LangChain object:
                   SystemMessage(content="...")   — instruction for the model (agent role)
                   HumanMessage(content="...")    — message from the user
                   AIMessage(...)                 — previous model response
                   ToolMessage(content="...", tool_call_id="...") — tool result

      tools   — list of tool descriptions (OpenAI function calling schema or LangChain tools).

    Returns AIMessage:
      msg.content    — text response (str)
      msg.tool_calls — list of tool calls:
                         "name" — tool name
                         "args" — arguments (already parsed dict)
                         "id"   — unique call identifier
    """
    if tools:
        return llm.bind_tools(tools).invoke(messages)
    return llm.invoke(messages)


# Product catalog
CATALOG = [
    {"id": "p1",  "name": "Sony WH-1000XM5",            "category": "headphones", "brand": "Sony",     "price": 349, "color": "black",    "rating": 4.8, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p2",  "name": "Sony WH-CH720N",              "category": "headphones", "brand": "Sony",     "price": 129, "color": "blue",     "rating": 4.4, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p3",  "name": "Bose QuietComfort Ultra",     "category": "headphones", "brand": "Bose",     "price": 379, "color": "white",    "rating": 4.7, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p4",  "name": "Apple AirPods Pro 2",         "category": "earbuds",    "brand": "Apple",    "price": 249, "color": "white",    "rating": 4.6, "tags": ["wireless", "noise-cancelling", "ios"]},
    {"id": "p5",  "name": "Anker Soundcore Liberty 4 NC","category": "earbuds",    "brand": "Anker",    "price": 99,  "color": "black",    "rating": 4.3, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p6",  "name": "Logitech MX Master 3S",       "category": "mouse",      "brand": "Logitech", "price": 109, "color": "graphite", "rating": 4.8, "tags": ["wireless", "productivity", "premium"]},
    {"id": "p7",  "name": "Logitech Pebble 2",           "category": "mouse",      "brand": "Logitech", "price": 34,  "color": "white",    "rating": 4.2, "tags": ["wireless", "budget", "portable"]},
    {"id": "p8",  "name": "Keychron K2",                 "category": "keyboard",   "brand": "Keychron", "price": 89,  "color": "black",    "rating": 4.5, "tags": ["wireless", "mechanical", "compact"]},
    {"id": "p9",  "name": "NuPhy Air75",                 "category": "keyboard",   "brand": "NuPhy",    "price": 139, "color": "gray",     "rating": 4.6, "tags": ["wireless", "mechanical", "low-profile"]},
    {"id": "p10", "name": "Amazon Kindle Paperwhite",    "category": "ereader",    "brand": "Amazon",   "price": 149, "color": "black",    "rating": 4.7, "tags": ["reading", "portable", "gift"]},
]


@dataclass
class ShopState:
    """Session state: cart and last search results."""
    cart: list = field(default_factory=list)
    last_results: list = field(default_factory=list)


@dataclass
class ToolCallRecord:
    name: str
    args: dict
    result: Any = None


class ToolTracer:
    """Collects all tool calls."""
    def __init__(self):
        self.calls: list[ToolCallRecord] = []

    def record(self, name: str, args: dict, result: Any = None) -> None:
        self.calls.append(ToolCallRecord(name=name, args=args, result=result))

    def called(self, name: str) -> bool:
        return any(c.name == name for c in self.calls)

    def get_calls(self, name: str) -> list:
        return [c for c in self.calls if c.name == name]

    def print_trace(self) -> None:
        print("=== Tool Call Trace ===")
        for i, c in enumerate(self.calls, 1):
            print(f"  {i}. {c.name}({json.dumps(c.args, ensure_ascii=False)[:80]})")
            if c.result is not None:
                print(f"     -> {json.dumps(c.result, ensure_ascii=False)[:100]}")
        print("=====================")


class ShopTools:
    """Shop logic — search and add to cart."""
    def __init__(self, catalog):
        self.catalog = catalog

    def search_products(self, query: str = "", category: str | None = None,
                        brand: str | None = None, max_price: float | None = None,
                        sort_by: str | None = None) -> list:
        results = []
        q_words = query.lower().split() if query else []
        for item in self.catalog:
            hay = f"{item['name']} {item['category']} {item['brand']} {' '.join(item['tags'])}".lower()
            if q_words and not all(w in hay for w in q_words): continue
            if category and item["category"] != category: continue
            if brand and item["brand"].lower() != brand.lower(): continue
            if max_price is not None and item["price"] > float(max_price): continue
            results.append(copy.deepcopy(item))
        if sort_by == "price_asc": results.sort(key=lambda x: x["price"])
        elif sort_by == "rating_desc": results.sort(key=lambda x: -x["rating"])
        return results

    def add_to_cart(self, state: ShopState, product_id: str, quantity: int = 1) -> dict:
        product = next((p for p in self.catalog if p["id"] == product_id), None)
        if not product:
            return {"ok": False, "error": f"Product {product_id} not found"}
        existing = next((r for r in state.cart if r["product_id"] == product_id), None)
        if existing:
            existing["quantity"] += quantity
        else:
            state.cart.append({"product_id": product_id, "name": product["name"],
                                "price": product["price"], "quantity": quantity})
        return {"ok": True, "cart_size": len(state.cart)}


@dataclass
class AgentContext:
    """Shared context passed between agents in Task 3."""
    query: str
    max_price: float | None = None
    candidates: list[dict] = field(default_factory=list)
    pros: dict[str, str] = field(default_factory=dict)   # product_id -> pros description
    cons: dict[str, str] = field(default_factory=dict)   # product_id -> cons description
    best: dict | None = None
    cart_result: dict | None = None


TOOLS = ShopTools(CATALOG)
print("Template loaded.")
print(f"  Model: {MODEL_NAME}")
print(f"  Catalog: {len(CATALOG)} products")
print(f"  Utilities: AgentContext, ToolTracer, ShopTools, convert_to_openai_tool")
print(f"  LangChain: HumanMessage, SystemMessage, AIMessage, ToolMessage")


/Users/romansafronenkov/Documents/Projects/venvs/llm_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Template loaded.
  Model: gpt://b1gqfjjlhfg3g261lqcg/gpt-oss-20b
  Catalog: 10 products
  Utilities: AgentContext, ToolTracer, ShopTools, convert_to_openai_tool
  LangChain: HumanMessage, SystemMessage, AIMessage, ToolMessage


In [3]:
from pydantic import BaseModel, Field
from typing import List, Literal

# ╔══════════════════════════════════════════════════════════════╗
# ║               YOUR CODE — THREE TASKS                        ║
# ╚══════════════════════════════════════════════════════════════╝

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 1. Tool-Calling Agent (ReAct loop)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 1.1. Define SHOP_TOOLS_SCHEMA — tool descriptions for the LLM.
#
# Below are stub functions with signatures but no descriptions.
# The LLM needs to understand what each tool does and what its parameters mean.
#
# Task: add a docstring (description + Args) to each function.
# The convert_to_openai_tool() function from the template will generate the JSON schema automatically.
# For docstring format details, see Google-style docstrings.

def search_products(
    query: str = "",
    category: str | None = None,
    brand: str | None = None,
    max_price: float | None = None,
    sort_by: str | None = None,
) -> list:
    # YOUR CODE HERE: add a docstring describing the tool and its parameters
    """
    Tool for searching the products.

    This function is looking for a product in catalogue, using the search query.
    Checks if any words from query matches 'item', 'category', 'brand' or 'tags' from catalogue.

    Args:
        query (str, default=""): The search query. Used for item search in catalogue
        category (str or None, default=None): The item category to filter the catalogue. Optional
        brand (str or None, default=None): The item brand to filter the catalogue. Optional
        max_price (float or None, default=None): The item max price to filter items. Optional
        sort_by (str or None, default=None): Parameter that used to return sorted products. Must be one of the following: 'price_asc': to sort products by price in the ascending order, 'rating_desc': to sort products by rating in the descending order.

    Returns:
        list: The result of the search, consists of the products from the catalogue: 
            [{"id": "...",  "name": "...", "category": "...", "brand": "...", "price": ..., "color": "...", "rating": ..., "tags": ["...", "...", "..."]}, ...].
    """
    pass

def add_to_cart(product_id: str, quantity: int = 1) -> dict:
    # YOUR CODE HERE: add a docstring
    """
    Tool for adding the product with product_id to the cart.

    Args:
        product_id (str): The id of the product to be added to the cart
        quantity (int, default=1): The quantity of the product, that should be added to the cart

    Returns:
        dict: If adding succeded:
            {"ok": True, "cart_size": ...}
            else:
            {"ok": False, "error": ...}
    """
    pass

# YOUR CODE HERE: generate the schema
SHOP_TOOLS_SCHEMA = [
    convert_to_openai_tool(search_products),
    convert_to_openai_tool(add_to_cart),
]


# 1.2. Implement run_shopping_agent — a ReAct shop agent.
def run_shopping_agent(user_message: str, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> str:
    """
    ReAct shop agent. Receives a user message and iteratively:
      1. Calls the LLM with the history and tool schema.
      2. If the LLM returns tool_calls — executes each tool:
           search_products -> saves result to state.last_results, records in tracer
           add_to_cart     -> adds product to state.cart, records in tracer
         Adds a ToolMessage with the result to the history and repeats the loop.
      3. If tool_calls is empty — returns the text response from the LLM.
    """
    
    SYSTEM_PROMPT = """You are an agent for an online electronics shop. Follow the TAO loop:
    
    THOUGHT: Analyze the request, plan the steps
    ACTION: Call the appropriate tool to get information
    OBSERVATION: Analyze the result, determine if further actions are needed
    
    Available tools:
    - search_products: search for products in catalogue
    - add_to_cart: add certain product to cart
    
    IMPORTANT:
    - Call only ONE tool per step. Get the result, analyze it, then decide what to do next
    - Do not fabricate information — only use data from tools
    - Be polite and helpful to the customer
    """
    n_max_steps = 30
    cur_step = 0
    messages = [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=user_message)]
    
    while cur_step != n_max_steps:
        cur_step += 1
        response = llm_chat(messages, tools=SHOP_TOOLS_SCHEMA)
        if hasattr(response, 'tool_calls') and response.tool_calls:
            for tool_call in response.tool_calls:
                tool_name = tool_call['name']
                tool_args = tool_call['args']
                if tool_name == 'add_to_cart':
                    tool_args['state'] = state
                    
                tool_call_result = tools.__getattribute__(tool_name)(**tool_args)
                if tool_name == 'search_products':
                    state.last_results = tool_call_result

                tracer.record(name=tool_name, args=tool_args, result=tool_call_result)
                messages.append(ToolMessage(content=json.dumps(tool_call_result), tool_call_id=tool_call['id']))
        if response.content and not response.tool_calls:
            return response.content


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 2. Memory Agent
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PROFILE_PATH = Path("user_profile.json")
# Recommended profile fields:
#   name       — user name
#   brand      — preferred brand
#   max_price  — maximum price
#   color      — preferred color
#   category   — category of interest

def load_profile(path: Path = PROFILE_PATH) -> dict:
    """Loads profile from JSON. Returns {} if file does not exist."""
    if not os.path.exists(path):
        return {}
        
    with open(path, 'r') as f:
        profile = json.load(f)
    return profile

def save_profile(profile: dict, path: Path = PROFILE_PATH) -> None:
    """Saves the profile dict to a file as JSON."""
    with open(path, 'w') as f:
        json.dump(profile, f)

def update_profile(key: str, value: any, path: Path = PROFILE_PATH) -> dict:
    """
    Tool for updating the client's profile.
    Takes the client's key:value and saves the updated profile to the file.
    Recommended field names: "name" - client's name, "brand" - client's favourite brand, "max_price" - max price for client, "color" - client's favourite colour, "category" - client's favourite category

    Args:
        key (str): The parameter of the client's profile (e.g. "name")
        value (any): The value of clients profile parameter (e.g. "Mike")

    Returns:
        dict: If adding succeded:
            {"ok": True, "key": ..., "value"}
            else:
            {"ok": False}
    """
    profile = load_profile(path)
    profile[key] = value
    try:
        save_profile(profile, path)
        return {"ok": True, "key": key, "value": value}
    except Exception as e:
        return {"ok": False}
    

SHOP_TOOLS_SCHEMA_WITH_MEMORY = SHOP_TOOLS_SCHEMA + [
    convert_to_openai_tool(update_profile)
    # YOUR CODE HERE — SHOP_TOOLS_SCHEMA + update_profile tool
    # update_profile: takes key (recommended: name | brand | max_price | color | category)
    #                 and value — saves a user preference to the profile
]

def run_memory_agent(
    user_message: str,
    state: ShopState,
    tools: ShopTools,
    tracer: ToolTracer,
    history: list,
    profile_path: Path = PROFILE_PATH,
) -> tuple:
    """
    Memory agent. Extends run_shopping_agent with long-term and short-term memory.

    Long-term memory:
      - Loads profile from file (load_profile) on each run
      - Passes profile to agent via SystemMessage
      - update_profile tool updates the profile on disk when preferences are first mentioned

    Short-term memory:
      - history contains the full message history from previous turns (including ToolMessages)
      - This allows the agent to "see" the results of past searches in the next turn
      - Added to the query before calling the LLM

    Returns (response: str, updated_history: list).
    Hint: save ALL messages to history (HumanMessage, AIMessage, ToolMessage),
    so the agent knows what was found in the next turn.
    """
    
    SYSTEM_PROMPT = """You are an agent for an online electronics shop.
    You may know the information about the client. If you see the new information, use the 'update_profile' tool to update it.
    
    Follow the TAO loop:
    
    THOUGHT: Analyze the request, plan the steps
    ACTION: Call the appropriate tool to get information
    OBSERVATION: Analyze the result, determine if further actions are needed
    
    Available tools:
    - search_products: search for products in catalogue
    - add_to_cart: add certain product to cart
    - update_profile: update client's profile (format: {"key": "...", "value": ...}). Update only one parameter per call.
    
    IMPORTANT:
    - Call only ONE tool per step. Get the result, analyze it, then decide what to do next
    - Do not fabricate information — only use data from tools
    - Be polite and helpful to the customer
    """

    SYSTEM_CLIENTS_PROFILE_PROMPT = """To provide the answer, use the information, that you know about the client:"""
    
    n_max_steps = 30
    cur_step = 0
    if not history:
        messages = [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=user_message)]
    else:
        messages = history + [HumanMessage(content=user_message)]
    
    while cur_step != n_max_steps:
        profile = json.dumps(load_profile(profile_path))
        messages += [SystemMessage(content=SYSTEM_CLIENTS_PROFILE_PROMPT+profile)]
        
        cur_step += 1
        response = llm_chat(messages, tools=SHOP_TOOLS_SCHEMA_WITH_MEMORY)
        if hasattr(response, 'tool_calls') and response.tool_calls:
            for tool_call in response.tool_calls:
                tool_name = tool_call['name']
                tool_args = tool_call['args']
                if tool_name == 'update_profile':
                    tool_args['path'] = profile_path
                    tool_call_result = update_profile(**tool_args)
                else:
                    if tool_name == 'add_to_cart':
                        tool_args['state'] = state
                        
                    tool_call_result = tools.__getattribute__(tool_name)(**tool_args)
                    if tool_name == 'search_products':
                        state.last_results = tool_call_result

                tracer.record(name=tool_name, args=tool_args, result=tool_call_result)
                messages.append(ToolMessage(content=json.dumps(tool_call_result), tool_call_id=tool_call['id']))
        if response.content and not response.tool_calls:
            return response.content, messages


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TASK 3. Multi-Agent System
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# Implement a system of four agents + an orchestrator.
# Goal — find the best product and honestly describe its pros and cons.
# Agents work in a chain via a shared AgentContext object (defined in the template).
#
# RetrieverAgent (LLM + tools)
#   Searches for up to 5 relevant products via search_products.
#   Fills ctx.candidates and ctx.max_price.
#   Important: only pass the search tool (not add_to_cart).
#
# ProsAgent (LLM, no tools)
#   For each product in ctx.candidates, writes 1-2 sentences of pros.
#   Fills ctx.pros (dict: product_id -> pros string).
#   Records an "analyze_pros" call in tracer.
#
# ConsAgent (LLM, no tools)
#   For each product in ctx.candidates, writes 1-2 sentences of cons.
#   Fills ctx.cons (dict: product_id -> cons string).
#   Records an "analyze_cons" call in tracer.
#
# RankerAgent (no LLM — logic only)
#   Picks the best product from ctx.candidates:
#     - Filters by ctx.max_price (if set)
#     - Among remaining: highest rating; if tied — lowest price
#   Records a "rank_candidates" call in tracer. Fills ctx.best.
#
# CoordinatorAgent (orchestrator)
#   Runs agents in a chain, maintains a trace list.
#   Trace keys: "delegate_retriever", "delegate_pros", "delegate_cons",
#               "delegate_ranker", "delegate_cart".
#   No CartAgent needed — if the user asks to add to cart,
#   CoordinatorAgent does it itself via tools.add_to_cart after ranking.
#   Returns AgentResult with response, trace, and context.
#   The response should include: product name, price, rating, pros and cons.

@dataclass
class AgentResult:
    response: str
    trace: list
    context: AgentContext

SHOP_TOOLS_SCHEMA_RETRIEVER = [
    convert_to_openai_tool(search_products)
]

class RetrieverAgent:
    def run(self, ctx: AgentContext, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> AgentContext:
        """Searches for products via LLM+tools. Fills ctx.candidates and ctx.max_price."""
        
        SYSTEM_PROMPT = """You are a Retriever agent for an online electronics shop.
        Your goal is to call the tool 'search_products' and search up to 5 relevant products.
        You will be provided with the products you have already chosen.
        If there are not enough candidates, call tool again, but change query to find more.
        
        Follow the TAO loop:
        
        THOUGHT: Analyze the request, plan the steps
        ACTION: Call the appropriate tool to get information
        OBSERVATION: Analyze the result, determine if further actions are needed
        
        Available tools:
        - search_products: search for products in catalogue
        
        IMPORTANT:
        - Call only ONE tool per step. Get the result, analyze it, then decide what to do next
        - Do not fabricate information — only use data from tools
        """
        
        user_message = ctx.query
        n_max_steps = 5
        cur_step = 0
        messages = [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=user_message)]
        candidates = []
        max_price = -1
        
        while cur_step != n_max_steps:
            cur_step += 1
            response = llm_chat(messages, tools=SHOP_TOOLS_SCHEMA_RETRIEVER)
            if hasattr(response, 'tool_calls') and response.tool_calls:
                for tool_call in response.tool_calls:
                    print(tool_call)
                    tool_name = tool_call['name']
                    tool_args = tool_call['args']
                        
                    tool_call_result = tools.__getattribute__(tool_name)(**tool_args)
                    if tool_name == 'search_products':
                        state.last_results = tool_call_result
                    candidates.extend(tool_call_result)
                    for res in tool_call_result:
                        if res['price'] > max_price:
                            max_price = res['price']
    
                    tracer.record(name=tool_name, args=tool_args, result=tool_call_result)
                    messages.append(ToolMessage(content=json.dumps(tool_call_result), tool_call_id=tool_call['id']))
            if response.content and not response.tool_calls:
                break
                
        ctx.candidates = candidates
        ctx.max_price = max_price
        return ctx


class ProsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Finds pros for each product via LLM. Fills ctx.pros."""
        
        SYSTEM_PROMPT = """You are a Pros agent for an online electronics shop.
        Your goal is to write in 1-2 sentences reasons why the customer should buy the product.
        Write only advantages of the product.
        """
        messages = [SystemMessage(content=SYSTEM_PROMPT)]
        pros = {}
        for candidate in ctx.candidates:
            query = f"Here is the short product description:{json.dumps(candidate)}"
            advantage = llm_chat(messages+[HumanMessage(content=query)])
            
            pros[candidate['id']] = advantage.content
        ctx.pros = pros
        tracer.record(name='analyze_pros', args={}, result=pros)
        
        return ctx


class ConsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        SYSTEM_PROMPT = """You are a Cons agent for an online electronics shop.
        Your goal is to write in 1-2 sentences reasons why the customer should not buy the product.
        Write only disadvantages of the product.
        """
        messages = [SystemMessage(content=SYSTEM_PROMPT)]
        cons = {}
        for candidate in ctx.candidates:
            query = f"Here is the short product description:{json.dumps(candidate)}"
            disadvantage = llm_chat(messages+[HumanMessage(content=query)])
            
            cons[candidate['id']] = disadvantage.content
        ctx.cons = cons
        tracer.record(name='analyze_cons', args={}, result=cons)
        
        return ctx


class RankerAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Picks the best product from ctx.candidates considering ctx.max_price. Fills ctx.best."""
        max_price = ctx.max_price or float('inf')
        best_candidate = None
        
        for candidate in ctx.candidates:
            if candidate['price'] > max_price:
                continue

            if not best_candidate:
                best_candidate = candidate
                
            if (best_candidate
                and candidate['rating'] > best_candidate['rating']):
                best_candidate = candidate

            if (best_candidate
                and candidate['rating'] == best_candidate['rating']
                and candidate['price'] < best_candidate['price']
               ):
                best_candidate = candidate
        
        ctx.best = best_candidate
        tracer.record(name='rank_candidates', args={}, result=best_candidate)
        return ctx

class SubTask(BaseModel):
    """Subtask for a specialized agent."""

    agent_name: Literal['retriever', 'pros_agent', 'cons_agent', 'ranker']
    description: str = Field(description='What needs to be done')
    priority: int = Field(ge=1, le=3, description='1=highest, 3=lowest')


class CoordinatorPlan(BaseModel):
    """Coordinator plan: decomposition into subtasks."""

    reasoning: str = Field(description='Why this decomposition was chosen')
    subtasks: List[SubTask] = Field(description='List of subtasks in execution order')

SHOP_TOOLS_SCHEMA_COORDINATOR = [
    convert_to_openai_tool(add_to_cart)
]

class CoordinatorAgent:
    def __init__(self):
        self.planning_llm = llm.with_structured_output(CoordinatorPlan)
        
        self.retriever = RetrieverAgent()
        self.pros_agent = ProsAgent()
        self.cons_agent = ConsAgent()
        self.ranker = RankerAgent()

        self.agents = {
            'retriever': self.retriever,
            'pros_agent': self.pros_agent,
            'cons_agent': self.cons_agent,
            'ranker': self.ranker
        }

    def run(self, user_message: str, state: ShopState, tools: ShopTools) -> AgentResult:
        """Orchestrates agents. Returns AgentResult with response, trace, and context."""
        
        SYSTEM_PROMPT = """You are the coordinator of an electronics store agent team.
        Your goal is to create execution plan and delegate task to specialists to find the best product for user request.
        Do not add any product to cart at this stage, it will be done later.
        
        You have 4 specialists:
        - retriever: search for product candidates, creates list of candidates, based on user's query
        - pros_agent: for each candidate create reasons to choose the certain product, writes advantages
        - cons_agent: for each candidate create reasons not to choose the certain product, writes disadvantages
        - ranker: picks the best product among the candidates
        
        Break the user's request into subtasks for the specialists.
        Specify priority (1=highest) and execution order.
        
        CRITICAL RULES for subtask descriptions:
        - pros_agent, cons_agent and ranker cannot work without candidates, created by the retriever.
        """
        ctx = AgentContext(query=user_message)
        tracer = ToolTracer()
        trace_keys = {
            'retriever': 'delegate_retriever',
            'pros_agent': 'delegate_pros',
            'cons_agent': 'delegate_cons',
            'ranker': 'delegate_ranker'
        }
        
        # create plan
        plan = self.planning_llm.invoke([SystemMessage(SYSTEM_PROMPT), HumanMessage(user_message)])

        print(plan)
        
        # execute plan
        sorted_tasks = sorted(plan.subtasks, key=lambda t: t.priority)
 
        for task in sorted_tasks:
            agent = self.agents.get(task.agent_name)
            if not agent:
                continue
            
            tracer.record(name=trace_keys[task.agent_name], args={}, result=None)

            if task.agent_name == 'retriever':
                ctx = agent.run(ctx=ctx, tracer=tracer, state=state, tools=tools)
            else:
                ctx = agent.run(ctx=ctx, tracer=tracer)

        # aggregate results
        SYSTEM_PROMPT_AGGREGATE = """You are the coordinator. Combine the specialists' results
        into a single coherent response for the customer. Be polite and informative.
        Do not repeat internal details, only include information useful to the customer.
        Use ONLY facts from the specialist results below. Do NOT invent or assume any data.
        
        If user wants to add the product to the cart, you can add do it using tool.

        Available tools:
        - add_to_cart: add certain product to cart.

        CRITICAL RULES:
        - The response should include: product name, price, rating, pros and cons.
        """

        messages = [
            SystemMessage(SYSTEM_PROMPT_AGGREGATE),
            HumanMessage(f'User request: {user_message}, Specialists results:\nretriever:{ctx.candidates}\npros_agent:{ctx.pros}\ncons_agent:{ctx.cons}\nranker:{ctx.best}\n')
        ]

        cur_step = 0
        n_max_steps = 5
        while cur_step != n_max_steps:
            cur_step += 1
            response = llm_chat(messages, tools=SHOP_TOOLS_SCHEMA_COORDINATOR)
            if hasattr(response, 'tool_calls') and response.tool_calls:
                for tool_call in response.tool_calls:
                    tool_name = tool_call['name']
                    tool_args = tool_call['args']
                    tool_args['state'] = state
                        
                    tool_call_result = tools.__getattribute__(tool_name)(**tool_args)
    
                    tracer.record(name='delegate_cart', args=tool_args, result=tool_call_result)
                    messages.append(ToolMessage(content=json.dumps(tool_call_result), tool_call_id=tool_call['id']))
            if response.content and not response.tool_calls:
                trace = [call.name for call in tracer.calls]
                result = AgentResult(response=response.content, trace=trace, context=ctx)
                return result
            

/Users/romansafronenkov/Documents/Projects/venvs/llm_venv/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:648: ArbitraryTypeWarning: <built-in function any> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warnings.warn(
/Users/romansafronenkov/Documents/Projects/venvs/llm_venv/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:648: ArbitraryTypeWarning: <built-in function any> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warnings.warn(


In [5]:
# --- Open examples for Task 1 -------------------------------------------

# [1.A] Search with price filter
_s1a = ShopState(); _t1a = ToolTracer()
_r1a = run_shopping_agent("Find wireless headphones under 150 dollars", _s1a, TOOLS, _t1a)
_t1a.print_trace()
assert _t1a.called("search_products"), "FAIL: search_products was not called"
assert all(p["price"] <= 150 for p in _s1a.last_results)
print("OK 1.A")

# [1.B] Search + add the cheapest
_s1b = ShopState(); _t1b = ToolTracer()
_r1b = run_shopping_agent(
    "Find a wireless mouse under 120 dollars and add the cheapest one to cart",
    _s1b, TOOLS, _t1b
)
assert _t1b.called("search_products") and _t1b.called("add_to_cart")
assert len(_s1b.cart) == 1 and _s1b.cart[0]["product_id"] == "p7"
print("OK 1.B")

# [1.C] Best keyboard
_s1c = ShopState(); _t1c = ToolTracer()
_r1c = run_shopping_agent(
    "Find a wireless keyboard with the best rating and add it to cart",
    _s1c, TOOLS, _t1c
)
assert _t1c.called("search_products") and _t1c.called("add_to_cart")
added = next(p for p in CATALOG if p["id"] == _s1c.cart[0]["product_id"])
assert added["category"] == "keyboard"
print(f"OK 1.C: '{added['name']}' (rating {added['rating']})")


=== Tool Call Trace ===
  1. search_products({"query": "wireless headphones", "category": "headphones", "max_price": 150, "so)
     -> [{"id": "p2", "name": "Sony WH-CH720N", "category": "headphones", "brand": "Sony", "price": 129, "co
OK 1.A
OK 1.B
OK 1.C: 'NuPhy Air75' (rating 4.6)


In [6]:
# --- Open examples for Task 2 -------------------------------------------

# [2.A] Saving preferences
_p2a = Path("_demo_profile_2a.json")
if _p2a.exists(): _p2a.unlink()
_s2a = ShopState(); _t2a = ToolTracer(); _h2a = []
_r2a, _h2a = run_memory_agent(
    "My name is Anna, I prefer Sony and my budget is 200 dollars",
    _s2a, TOOLS, _t2a, _h2a, _p2a
)
_prof2a = load_profile(_p2a)
assert _t2a.called("update_profile") and _prof2a.get("brand") == "Sony"
print("OK 2.A")

# [2.B] New session uses profile (history=[])
_p2b = Path("_demo_profile_2b.json")
save_profile({"name": "Boris", "brand": "Logitech", "max_price": "150"}, _p2b)
_s2b = ShopState(); _t2b = ToolTracer(); _h2b = []
_r2b, _ = run_memory_agent("What is my name and what is my budget?", _s2b, TOOLS, _t2b, _h2b, _p2b)
assert "Boris" in _r2b
print("OK 2.B")

# [2.C] Short-term memory — turn 2 remembers turn 1
_p2c = Path("_demo_profile_2c.json")
if _p2c.exists(): _p2c.unlink()
_s2c = ShopState(); _h2c = []
_, _h2c = run_memory_agent(
    "Find wireless headphones under 150 dollars", _s2c, TOOLS, ToolTracer(), _h2c, _p2c
)
assert len(_h2c) >= 2
_t2c2 = ToolTracer()
_, _h2c = run_memory_agent(
    "Add the first one found to cart", _s2c, TOOLS, _t2c2, _h2c, _p2c
)
assert _t2c2.called("add_to_cart") and len(_s2c.cart) == 1
print(f"OK 2.C: added '{_s2c.cart[0]['name']}'")


OK 2.A
OK 2.B
OK 2.C: added 'Sony WH-CH720N'


In [7]:
# --- Open examples for Task 3 -------------------------------------------

# [3.A] Full cycle: search -> pros -> cons -> ranking -> cart
_s3a = ShopState()
_res3a = CoordinatorAgent().run(
    "Find the best wireless mouse under 120 dollars and add it to cart", _s3a, TOOLS
)
assert "delegate_retriever" in _res3a.trace
assert "delegate_pros" in _res3a.trace and "delegate_cons" in _res3a.trace
assert "delegate_ranker" in _res3a.trace and "delegate_cart" in _res3a.trace
assert len(_s3a.cart) == 1 and _s3a.cart[0]["product_id"] == "p6"
assert _res3a.context.best is not None and _res3a.context.best["id"] == "p6"
assert len(_res3a.context.pros) > 0 and len(_res3a.context.cons) > 0
print("OK 3.A")

# [3.B] Search only, no add to cart
_s3b = ShopState()
_res3b = CoordinatorAgent().run("Find a wireless keyboard", _s3b, TOOLS)
assert "delegate_retriever" in _res3b.trace
assert "delegate_pros" in _res3b.trace and "delegate_cons" in _res3b.trace
assert "delegate_ranker" in _res3b.trace
assert "delegate_cart" not in _res3b.trace and len(_s3b.cart) == 0
assert _res3b.context.best is not None
print("OK 3.B")

# [3.C] RankerAgent — price tie-break with equal rating
_ctx3c = AgentContext(query="test", candidates=[
    {"id": "x1", "name": "A", "price": 200, "rating": 4.8},
    {"id": "x2", "name": "B", "price": 150, "rating": 4.8},
    {"id": "x3", "name": "C", "price": 100, "rating": 4.5},
])
_tr3c = ToolTracer()
_ctx3c = RankerAgent().run(_ctx3c, _tr3c)
assert _ctx3c.best["id"] == "x2" and _tr3c.called("rank_candidates")
print("OK 3.C")

# [3.D] RankerAgent respects ctx.max_price
_ctx3d = AgentContext(
    query="mouse under 120 dollars",
    max_price=120.0,
    candidates=[
        {"id": "expensive", "name": "Super Mouse",  "price": 200, "rating": 4.9},
        {"id": "p6",        "name": "MX Master 3S", "price": 109, "rating": 4.8},
        {"id": "p7",        "name": "Pebble 2",      "price": 34,  "rating": 4.2},
    ],
)
_tr3d = ToolTracer()
_ctx3d = RankerAgent().run(_ctx3d, _tr3d)
assert _ctx3d.best is not None and _ctx3d.best["id"] == "p6"
print("OK 3.D: context passed correctly, max_price is respected")


reasoning='The user wants the best wireless mouse under $120. According to the system instructions, we must first gather candidate products, then evaluate pros and cons, then rank them, and only after the best product is identified do we add it to the cart. We’ll delegate each step to the appropriate specialist and specify the priority and execution order. The final step of adding to the cart is noted but not executed yet, as per the rule. The plan is presented in a clear, numbered list for the team. No product is added to the cart at this stage.' subtasks=[SubTask(agent_name='retriever', description='Search for wireless mice priced under $120. Compile a list of at least 5 distinct candidates, including brand, model, price, key specs (e.g., DPI, battery life, connectivity), and a short note on why each is a potential fit.', priority=1), SubTask(agent_name='pros_agent', description='For each candidate from the retriever, list the advantages that make it a strong choice for the user’s ne